# 🩺 Sprint 1 — Prétraitement des données
### Projet : Prédiction du risque de diabète — BRFSS 2015
---
**Formation** : Master IA — CESI  
**Dataset** : `diabetes_012_health_indicators_BRFSS2015.csv`  
**Livrable** : Sprint 1 — Préparation des données  

---

## 📋 Sommaire

1. [Contexte](#contexte)
2. [Objectif](#objectif)
3. [Problématique](#problématique)
4. [Contraintes](#contraintes)
5. [Étape 1 — Chargement et compréhension initiale](#etape1)
6. [Étape 2 — Séparation cible / variables explicatives](#etape2)
7. [Étape 3 — Découpage train / val / test](#etape3)
8. [Étape 4 — Typage des variables](#etape4)
9. [Étape 5 — Doublons et valeurs manquantes](#etape5)
10. [Étape 6 — Analyse exploratoire quantitative](#etape6)
11. [Étape 7 — Analyse exploratoire qualitative (EDA visuelle)](#etape7)
12. [Étape 8 — Normalisation](#etape8)
13. [Étape 9 — Sauvegarde](#etape9)
14. [Analyse RGPD & Éthique](#rgpd)

---
<a id="contexte"></a>
## 🌍 Contexte

Le diabète est l'une des maladies chroniques les plus répandues aux États-Unis. En 2018, les CDC estimaient à **34,2 millions** le nombre d'Américains diabétiques, et à **88 millions** ceux en état de prédiabète — dont une grande majorité ignorait leur situation.

Cette maladie se traduit par une mauvaise régulation du sucre sanguin, ce qui, sur le long terme, peut engendrer des complications graves : maladies cardiovasculaires, insuffisance rénale, amputations, perte de la vue. Son coût économique est estimé à près de **400 milliards de dollars par an**.

Le dataset utilisé provient du **Behavioral Risk Factor Surveillance System (BRFSS)**, une enquête téléphonique annuelle menée par les CDC auprès de plus de 400 000 Américains. Notre version (2015) contient **253 680 réponses** et **22 variables** relatives aux comportements de santé, antécédents médicaux et conditions socio-économiques des participants.

---
<a id="objectif"></a>
## 🎯 Objectif

Construire un **modèle de classification binaire** capable de prédire si une personne est diabétique à partir de ses indicateurs de santé.

La variable cible `Diabetes_012` est **binarisée** comme suit :
- **0** → pas de diabète ou prédiabète (classes 0 et 1 regroupées)
- **1** → diabète avéré (classe 2)

La **métrique finale** à maximiser est la **ROC AUC** sur un jeu de test non étiqueté.

---
<a id="problematique"></a>
## ❓ Problématique

> *À partir des indicateurs de santé comportementaux et médicaux recueillis dans le BRFSS 2015, est-il possible de prédire avec fiabilité si une personne souffre de diabète ?*

Ce sprint se concentre uniquement sur la **préparation des données** : nettoyage, exploration, et normalisation. Les modèles seront construits dans les sprints suivants.

---
<a id="contraintes"></a>
## ⚙️ Contraintes

| Contrainte | Détail |
|---|---|
| **Déséquilibre des classes** | La classe diabète représente ~14% du dataset |
| **Qualité des données** | Présence de doublons (~24 000) à gérer |
| **RGPD** | Variables de santé relevant de l'art. 9 RGPD — traitement justifié par l'art. 9.2.j |
| **MLOps** | Scaler ajusté uniquement sur le train (pas de data leakage) |
| **Reproductibilité** | `random_state=42` sur tous les découpages |

---
## 📦 Imports et configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from pathlib import Path

# Style général des graphiques
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

# Répertoires de sortie
PROCESSED_DIR = Path("data/processed")
FIGURES_DIR   = Path("reports/figures")
REPORT_DIR    = Path("reports/sprint1")
for d in [PROCESSED_DIR, FIGURES_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print("✅ Imports et configuration OK")

---
<a id="etape1"></a>
## Étape 1 — Chargement et compréhension initiale du dataset

On commence par charger le CSV brut et regarder ce qu'on a : dimensions, types, premières lignes, statistiques descriptives et valeurs manquantes.

In [ ]:
# Chargement du CSV brut
CSV_PATH = "diabetes_012_health_indicators_BRFSS2015.csv"
df_raw = pd.read_csv(CSV_PATH)

print(f"Shape : {df_raw.shape[0]:,} lignes x {df_raw.shape[1]} colonnes")
print(f"Mémoire estimée : {df_raw.memory_usage(deep=True).sum() / 1e6:.2f} Mo")
df_raw.head()

In [ ]:
# Types de données et valeurs manquantes
info_df = pd.DataFrame({
    "dtype"    : df_raw.dtypes,
    "n_unique" : df_raw.nunique(),
    "n_missing": df_raw.isnull().sum(),
    "pct_miss" : (df_raw.isnull().mean() * 100).round(2)
})
print(info_df.to_string())
print(f"\nTotal valeurs manquantes : {df_raw.isnull().sum().sum()}")

In [ ]:
# Statistiques descriptives
df_raw.describe().T.round(2)

**Observations :**
- Le dataset contient **253 680 lignes** et **22 variables**, toutes de type `float64`.
- **Aucune valeur manquante** — le BRFSS encode les non-réponses par des codes spécifiques déjà filtrés par le CDC.
- La variable cible `Diabetes_012` prend les valeurs 0, 1 (prédiabète) ou 2 (diabète).
- On note la présence de **doublons** à traiter dans l'étape 5.

---
<a id="etape2"></a>
## Étape 2 — Séparation cible / variables explicatives

On isole la variable cible `Diabetes_012` et on la **binarise** : les classes 0 (pas de diabète) et 1 (prédiabète) sont regroupées sous la valeur 0. La classe 2 (diabète avéré) devient 1.

In [ ]:
# Distribution initiale de la cible
print("Distribution Diabetes_012 avant binarisation :")
print(df_raw["Diabetes_012"].value_counts().sort_index())

# Binarisation
df = df_raw.copy()
df["Diabetes_binary"] = (df["Diabetes_012"] == 2).astype(int)
df = df.drop(columns=["Diabetes_012"])

print("\nDistribution après binarisation :")
print(df["Diabetes_binary"].value_counts().sort_index())
print(f"Taux positif : {df['Diabetes_binary'].mean():.3%}")

In [ ]:
# Séparation X / y
TARGET = "Diabetes_binary"
FEATURES = [c for c in df.columns if c != TARGET]

X_full = df[FEATURES]
y_full = df[TARGET]

print(f"Variables explicatives : {len(FEATURES)}")
print(f"Variable cible         : {TARGET}")
print(f"Shape X : {X_full.shape}  |  Shape y : {y_full.shape}")

---
<a id="etape3"></a>
## Étape 3 — Découpage train / validation / test

On effectue le découpage **avant** le nettoyage approfondi pour éviter tout data leakage (les statistiques de normalisation ne doivent pas "voir" le test).

Le découpage est **stratifié** sur la variable cible pour conserver le même ratio de positifs dans chaque partition.

- **Train** : 70%  
- **Validation** : 15%  
- **Test** : 15%

In [ ]:
# Premier split : train (70%) vs temp (30%)
X_train_raw, X_temp, y_train_raw, y_temp = train_test_split(
    X_full, y_full,
    test_size=0.30,
    stratify=y_full,
    random_state=RANDOM_STATE
)

# Deuxième split : val (15%) vs test (15%)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

for name, X, y in [("Train", X_train_raw, y_train_raw),
                   ("Val  ", X_val_raw,   y_val),
                   ("Test ", X_test_raw,  y_test)]:
    print(f"{name} → {len(X):>7,} lignes  |  taux positif : {y.mean():.3%}")

---
<a id="etape4"></a>
## Étape 4 — Typage des variables

On distingue deux catégories de variables :
- **Variables continues / ordinales** : BMI, MentHlth, PhysHlth, GenHlth, Age, Education, Income — seront standardisées.
- **Variables binaires** (0/1) : les 14 autres — conservées en l'état.

In [ ]:
# Typage explicite
BINARY_COLS = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"
]
NUMERIC_COLS = ["BMI", "MentHlth", "PhysHlth", "GenHlth", "Age", "Education", "Income"]

# Application sur le dataset complet (avant split)
df[BINARY_COLS] = df[BINARY_COLS].astype("int8")
df[NUMERIC_COLS] = df[NUMERIC_COLS].astype("float32")

print("Variables binaires  :", BINARY_COLS)
print()
print("Variables numériques:", NUMERIC_COLS)
print()
print(df.dtypes)

---
<a id="etape5"></a>
## Étape 5 — Nettoyage des doublons et gestion des valeurs manquantes

### 5.1 Valeurs manquantes

In [ ]:
# Valeurs manquantes
n_missing = df.isnull().sum().sum()
print(f"Total valeurs manquantes : {n_missing}")
print("→ Aucun traitement nécessaire. Le BRFSS ne contient pas de NA dans cette version.")

### 5.2 Doublons

Un doublon exact correspond à deux répondants ayant exactement le même profil sur les 21 variables. Étant donné le nombre important de variables binaires, de tels doublons sont plausibles statistiquement et on les supprime.

In [ ]:
# Doublons
n_before = len(df)
n_dup = df.duplicated().sum()
print(f"Doublons détectés : {n_dup:,}")

df_clean = df.drop_duplicates().reset_index(drop=True)
n_after = len(df_clean)
print(f"Lignes avant : {n_before:,}  →  après : {n_after:,}  (supprimés : {n_before - n_after:,})")

# Distribution de la cible après nettoyage
print("\nDistribution cible après nettoyage :")
print(df_clean["Diabetes_binary"].value_counts().sort_index())
print(f"Taux positif : {df_clean['Diabetes_binary'].mean():.3%}")

**Note RGPD/Éthique :** La suppression des doublons exacts est une mesure de **minimisation des données** conforme à l'art. 5.1.c du RGPD — on évite de surestimer le poids de certains profils.

---
<a id="etape6"></a>
## Étape 6 — Analyse exploratoire quantitative

On analyse les distributions, le déséquilibre des classes et les corrélations avec la variable cible.

In [ ]:
# Statistiques descriptives sur le jeu nettoyé
df_clean.describe().T.round(2)

In [ ]:
# Déséquilibre des classes
counts = df_clean["Diabetes_binary"].value_counts().sort_index()
ratios = counts / counts.sum()
imbalance_ratio = counts[0] / counts[1]

print("Effectifs :")
print(f"  Classe 0 (no diabète / prédiabète) : {counts[0]:,}  ({ratios[0]:.2%})")
print(f"  Classe 1 (diabète)                 : {counts[1]:,}  ({ratios[1]:.2%})")
print(f"\nImbalance ratio (majoritaire / minoritaire) : {imbalance_ratio:.2f}")

In [ ]:
# Top corrélations avec la cible
corr = df_clean.drop(columns=["Diabetes_binary"]).corrwith(df_clean["Diabetes_binary"])
corr_sorted = corr.abs().sort_values(ascending=False)

print("Top 10 corrélations (valeur absolue) avec Diabetes_binary :")
for feat, val in zip(corr_sorted.index[:10], corr.loc[corr_sorted.index[:10]]):
    bar = "█" * int(abs(val) * 40)
    print(f"  {feat:<25s} {val:+.3f}  {bar}")

---
<a id="etape7"></a>
## Étape 7 — Analyse exploratoire qualitative (EDA visuelle)

In [ ]:
# Figure 1 : Déséquilibre des classes
fig, ax = plt.subplots(figsize=(6, 4))
counts_plot = df_clean["Diabetes_binary"].value_counts().sort_index()
bars = ax.bar(["Classe 0\n(no diabète / prédiabète)", "Classe 1\n(diabète)"],
              counts_plot.values,
              color=["#4C9BE8", "#E8654C"], edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, counts_plot.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{val:,}\n({val/counts_plot.sum():.1%})",
            ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title("Déséquilibre des classes — Diabetes_binary", fontsize=13, fontweight="bold")
ax.set_ylabel("Nombre d'observations")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure 1 sauvegardée")

In [ ]:
# Figure 2 : Heatmap de corrélation
fig, ax = plt.subplots(figsize=(14, 11))
corr_matrix = df_clean.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 7})
ax.set_title("Matrice de corrélation — Dataset nettoyé", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure 2 sauvegardée")

In [ ]:
# Figure 3 : Distributions des variables numériques
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(NUMERIC_COLS):
    axes[i].hist(df_clean[col], bins=30, color="#4C9BE8", edgecolor="white", alpha=0.85)
    axes[i].set_title(col, fontsize=11, fontweight="bold")
    axes[i].set_ylabel("Fréquence")
# Masquer le dernier subplot vide
axes[-1].set_visible(False)
fig.suptitle("Distribution des variables numériques / ordinales", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_numeric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure 3 sauvegardée")

In [ ]:
# Figure 4 : Boxplots par classe pour les variables numériques
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
palette = {0: "#4C9BE8", 1: "#E8654C"}
for i, col in enumerate(NUMERIC_COLS):
    sns.boxplot(data=df_clean, x="Diabetes_binary", y=col,
                palette=palette, ax=axes[i], width=0.5)
    axes[i].set_title(col, fontsize=11, fontweight="bold")
    axes[i].set_xlabel("Diabetes_binary")
axes[-1].set_visible(False)
fig.suptitle("Distribution par classe — Variables numériques", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_boxplots_by_class.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure 4 sauvegardée")

In [ ]:
# Figure 5 : Prévalence du diabète par variable binaire
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(BINARY_COLS):
    prev = df_clean.groupby(col)["Diabetes_binary"].mean() * 100
    prev.plot(kind="bar", ax=axes[i], color=["#4C9BE8", "#E8654C"],
              edgecolor="white", width=0.6)
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].set_ylabel("% diabète")
    axes[i].set_xlabel("")
    axes[i].set_xticklabels(["Non (0)", "Oui (1)"], rotation=0, fontsize=8)
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
for j in range(len(BINARY_COLS), len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Prévalence du diabète selon les variables binaires", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_binary_prevalence.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure 5 sauvegardée")

In [ ]:
# Figure 6 : Corrélations avec la cible (barh)
fig, ax = plt.subplots(figsize=(9, 7))
corr_target = df_clean.drop(columns=["Diabetes_binary"]).corrwith(df_clean["Diabetes_binary"])
corr_target_sorted = corr_target.sort_values()
colors = ["#E8654C" if v > 0 else "#4C9BE8" for v in corr_target_sorted]
ax.barh(corr_target_sorted.index, corr_target_sorted.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Corrélation de chaque variable avec Diabetes_binary", fontsize=13, fontweight="bold")
ax.set_xlabel("Coefficient de corrélation de Pearson")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_correlation_with_target.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure 6 sauvegardée")

**Lecture des figures :**
- **Fig. 1** : Le dataset est déséquilibré — ~85% de non-diabétiques vs ~15% de diabétiques. C'est un point critique pour le choix du modèle (Sprint 2).
- **Fig. 2** : Peu de corrélations fortes entre variables explicatives — pas de multicolinéarité évidente à supprimer.
- **Fig. 3 & 4** : BMI, GenHlth et Age montrent des distributions clairement différentes entre les deux classes.
- **Fig. 5** : HighBP, HighChol, DiffWalk et HeartDiseaseorAttack sont fortement associés au diabète.
- **Fig. 6** : GenHlth, HighBP, BMI et DiffWalk sont les variables les plus corrélées à la cible.

---
<a id="etape8"></a>
## Étape 8 — Normalisation

On applique un **StandardScaler** (centrage-réduction) uniquement sur les variables continues/ordinales. Les variables binaires restent en l'état.

⚠️ Le scaler est **ajusté uniquement sur le train** et appliqué ensuite sur val et test — principe fondamental du MLOps pour éviter toute fuite d'information.

In [ ]:
# Re-découpage à partir du dataset nettoyé (df_clean)
X_clean = df_clean.drop(columns=["Diabetes_binary"])
y_clean = df_clean["Diabetes_binary"]

X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X_clean, y_clean, test_size=0.30, stratify=y_clean, random_state=RANDOM_STATE
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print("Découpage final (après nettoyage) :")
for name, X, y in [("Train", X_train_raw, y_train),
                   ("Val  ", X_val_raw,   y_val),
                   ("Test ", X_test_raw,  y_test)]:
    print(f"  {name} → {len(X):>7,} lignes  |  taux positif : {y.mean():.3%}")

In [ ]:
# Standardisation
scaler = StandardScaler()

# Fit UNIQUEMENT sur le train
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[NUMERIC_COLS] = scaler.fit_transform(X_train_raw[NUMERIC_COLS])
X_val[NUMERIC_COLS]   = scaler.transform(X_val_raw[NUMERIC_COLS])
X_test[NUMERIC_COLS]  = scaler.transform(X_test_raw[NUMERIC_COLS])

print(f"Colonnes standardisées : {NUMERIC_COLS}")
print(f"\nMoyennes sur le train après scaling (doivent être ~0) :")
print(X_train[NUMERIC_COLS].mean().round(4).to_string())
print(f"\nÉcarts-types sur le train après scaling (doivent être ~1) :")
print(X_train[NUMERIC_COLS].std().round(4).to_string())

In [ ]:
# Sauvegarde du scaler
scaler_path = PROCESSED_DIR / "scaler.joblib"
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler sauvegardé : {scaler_path}")

---
<a id="etape9"></a>
## Étape 9 — Sauvegarde du jeu de données nettoyé

In [ ]:
# Sauvegarde de tous les datasets
datasets = {
    "diabetes_clean_full.csv" : df_clean,
    "X_train.csv"             : X_train,
    "X_val.csv"               : X_val,
    "X_test.csv"              : X_test,
    "y_train.csv"             : y_train.to_frame(),
    "y_val.csv"               : y_val.to_frame(),
    "y_test.csv"              : y_test.to_frame(),
}

for filename, dataset in datasets.items():
    path = PROCESSED_DIR / filename
    dataset.to_csv(path, index=False)
    print(f"  ✅ {filename:<35s} ({len(dataset):,} lignes)")

print(f"\n📁 Tous les fichiers sont dans : {PROCESSED_DIR}/")

In [ ]:
# Récapitulatif final
print("=" * 55)
print("         RÉCAPITULATIF SPRINT 1")
print("=" * 55)
print(f"Dataset brut       : {df_raw.shape[0]:,} x {df_raw.shape[1]}")
print(f"Après nettoyage    : {df_clean.shape[0]:,} x {df_clean.shape[1]}")
print(f"Doublons supprimés : {df_raw.shape[0] - df_clean.shape[0]:,}")
print(f"Valeurs manquantes : 0")
print(f"Taux positif       : {df_clean['Diabetes_binary'].mean():.3%}")
print(f"Imbalance ratio    : {imbalance_ratio:.2f}")
print("-" * 55)
print(f"Train : {len(X_train):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}")
print(f"Scaler : StandardScaler (fit sur train uniquement)")
print("=" * 55)

---
<a id="rgpd"></a>
## 🔒 Analyse RGPD & Éthique

### Cadre légal

Ce projet manipule des **données de santé** au sens de l'art. 9 du RGPD (UE 2016/679). Le traitement est légitimé par l'**art. 9.2.j** qui autorise l'utilisation de telles données à des fins de recherche scientifique d'intérêt public, sous garanties appropriées.

### Anonymisation source

Le dataset BRFSS 2015 a déjà été anonymisé par les CDC avant publication :
- Aucun identifiant direct (pas de nom, adresse, téléphone, date de naissance précise)
- L'âge est regroupé en **13 tranches** — pas l'âge exact
- L'origine ethnique et la zone géographique fine ont été retirées
- Le risque de ré-identification par croisement reste faible étant donné la granularité restreinte

### Inventaire des variables sensibles

| Variable | Sensibilité | Décision |
|---|---|---|
| `HighBP`, `HighChol`, `Stroke`, `HeartDiseaseorAttack` | Santé (art. 9) | ✅ Conservées — facteurs de risque centraux |
| `BMI`, `MentHlth`, `PhysHlth`, `GenHlth` | Santé (art. 9) | ✅ Conservées — indicateurs cliniques |
| `Smoker`, `HvyAlcoholConsump`, `PhysActivity` | Mode de vie | ✅ Conservées — comportements à risque |
| `Sex`, `Age` | Personnel | ✅ Conservées — facteurs cliniques reconnus, non identifiants |
| `Income`, `Education` | Socio-économique | ✅ Conservées **avec vigilance** — audit du biais prévu Sprint 3 |
| Origine ethnique, zone géographique | Non présentes | — Absentes par construction CDC |

### Mesures opérationnelles

- **Minimisation** : aucun croisement de variables ne reconstruit un identifiant indirect
- **Reproductibilité** : `random_state=42` sur tous les splits — décisions traçables
- **Pas de data leakage** : scaler ajusté uniquement sur le train, sauvegardé via joblib
- **Données conservées localement** — aucun envoi vers un service externe

### Points de vigilance pour les sprints suivants

- **Sprint 2** : le déséquilibre des classes (~5.5 : 1) doit être traité — l'accuracy seule masquerait les faux négatifs sur la classe diabète
- **Sprint 3** : audit du biais par sous-groupe (Income, Sex, Age) sur le test set — vérifier que le modèle ne discrimine pas certaines populations
- **Rappel** : ce modèle **n'est pas un dispositif médical** (règlement DM EU 2017/745) et ne peut pas servir de décision automatisée individuelle (art. 22 RGPD)

---
## ✅ Sprint 1 terminé

Toutes les étapes du livrable 1 sont complètes :

1. ✅ Chargement et compréhension initiale  
2. ✅ Séparation cible / variables explicatives + binarisation  
3. ✅ Découpage train / val / test stratifié  
4. ✅ Typage des variables  
5. ✅ Nettoyage des doublons et valeurs manquantes  
6. ✅ Analyse exploratoire quantitative  
7. ✅ Analyse exploratoire qualitative (6 figures)  
8. ✅ Normalisation StandardScaler (fit sur train uniquement)  
9. ✅ Sauvegarde des datasets nettoyés  

**Prochain sprint →** Construction et évaluation des modèles de classification.